# 06 — Parâmetros do Dossel Urbano (UCP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/06_urban_canopy_parameters.pt.ipynb)

**Objetivo de aprendizagem.** Ao final deste notebook você usará `lcz_get_ucp` para baixar e processar
Parâmetros do Dossel Urbano (Urban Canopy Parameters, UCP) — densidade/altura/volume construído (GHSL),
morfologia de edificações e métricas de fração de terreno (WUDAPT/WUMPOD), cobertura vegetal/
impermeável, e rugosidade direcional de cânions urbanos — seja como pilha raster distribuída
espacialmente sobre uma cidade, seja extraída em coordenadas de pontos específicos (por exemplo,
estações meteorológicas), e entenderá por que essa extração pontual é o elo perdido entre a forma
urbana derivada de sensoriamento remoto e a análise climática baseada em estações.

## Da classe LCZ à forma física urbana: o que os UCPs acrescentam

O esquema LCZ (Stewart & Oke, 2012) é deliberadamente compacto: classifica cada superfície urbana e
natural em uma de 17 classes discretas com base em um punhado de propriedades morfológicas e de
cobertura do solo *típicas* (altura de edificação, espaçamento entre edificações/árvores,
impermeabilização, fator de visão do céu). Essa discretização é o que torna o LCZ tão útil como uma
unidade de referência comum entre cidades — mas também descarta informação. Dois locais mapeados como
"LCZ 2 — midrise compacto" podem ter alturas de edificação, frações impermeáveis ou cobertura vegetal
bem diferentes na prática; a classe LCZ só informa a *faixa típica*, não o valor *pixel a pixel* em um
ponto específico.

Os **Parâmetros do Dossel Urbano (UCPs)** são as variáveis contínuas e fisicamente embasadas que
preenchem essa lacuna. Enquanto `lcz_get_parameters` do LCZ4r (notebook 03) deriva 34 parâmetros
morfológicos analiticamente a partir da própria classe LCZ (por exemplo, "LCZ 2 tem fração de
superfície construída típica de 0,4-0,6"), `lcz_get_ucp` vai um nível mais fundo e busca dados de forma
urbana **diretamente observados por sensoriamento remoto** para a área exata do seu recorte de estudo,
independentemente da classe LCZ em que cada pixel se enquadra. Quatro famílias de UCPs são
processadas:

- **Densidade/altura/volume construído (GHSL)** — o Global Human Settlement Layer (Melchiorri et al.,
  2025, JRC) fornece *fração de superfície construída* (m²/m², quanto de um pixel é coberto por
  edificações), *altura de edificação* (m, acima do solo), *volume construído* (m³, altura × área de
  base — um proxy para massa/capacidade térmica total das edificações) e *densidade populacional*
  (pessoas/km²). Juntas, essas quatro variáveis capturam a pergunta "quanta cidade há aqui, e quão
  alta/densa ela é", que impulsiona tanto a capacidade de armazenamento de calor urbano quanto a carga
  de calor antropogênico de uma vizinhança.
- **Morfologia WUDAPT/WUMPOD (Patel & Roth, 2022/2024)** — fração de edificação, altura, comprimento,
  perímetro, fração de terreno, além de classificações de cobertura do solo (CGLC, fração urbana
  global). Esses dados fornecem as entradas geométricas (`índice de área de planta`, `razão entre
  superfície construída e área de planta`) exigidas pelos modelos clássicos de dossel urbano e de
  balanço de energia urbano de camada única.
- **Cobertura vegetal/impermeável (GLC_FCS30D, Zhang et al., 2021)** — % de cobertura arbórea e % de
  superfície impermeável a 30 m, uma medição direta do equilíbrio fração-vegetada/fração-impermeável
  que é o principal fator físico da intensidade diurna da Ilha de Calor Urbana.
- **Rugosidade direcional (cânion urbano)** — fração de terreno, índice de altura e parâmetros de
  rugosidade aerodinâmica (deslocamento de plano-zero `zdm`, comprimento de rugosidade `zdr`/`zom`/
  `zor`) calculados em quatro direções cardeais (0°, 45°, 90°, 135°). Esses descrevem como a geometria
  das edificações varia conforme a direção de aproximação do vento — essencial para modelar ventilação
  de cânions urbanos, dispersão de poluentes e o arrasto aerodinâmico que uma cidade impõe ao campo de
  vento da camada limite.

**Por que extrair UCPs *em locais de estações*, especificamente?** A leitura bruta de temperatura de
uma estação meteorológica é produto tanto da atmosfera quanto do que está imediatamente ao redor do
sensor: uma estação em um pequeno parque cercado por edifícios altos registrará valores diferentes de
uma em uma rua residencial aberta e de baixa altura, mesmo sob forçante regional idêntica. Se você
souber apenas a classe LCZ de uma estação, saberá a *categoria* de seu entorno; se, além disso, extrair
sua altura de edificação, fração impermeável e densidade de dossel (que é exatamente o que
`lcz_get_ucp(..., stations=...)` faz), você pode construir um modelo *quantitativo e fisicamente
interpretável* de por que a estação A é consistentemente 2°C mais quente que a estação B. Esse é
precisamente o conjunto de atributos que o notebook de interpolação espacial por machine learning
(`notebooks/local/05_ml_interpolation_ucp`) usa para treinar um Random Forest e prever temperatura
pixel a pixel em toda a grade de uma cidade, usando UCPs como preditores em vez de (ou junto com)
coordenadas espaciais simples. `lcz_get_ucp` é a etapa de aquisição de dados que alimenta esse
modelo.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1. Obter o mapa LCZ

Como em todo notebook desta série, começamos com `lcz_get_map`, que geocodifica o nome de uma cidade,
baixa apenas os pixels que a cobrem a partir do GeoTIFF Cloud-Optimized global do Zenodo, e retorna um
caminho local. Usamos novamente Juiz de Fora — sua área minúscula mantém todo download deste notebook
rápido.

In [2]:
from LCZ4py import lcz_get_map

lcz_map_path = lcz_get_map(city="Juiz de Fora")
print(lcz_map_path)

13:34:28 - LCZ4py._internal._lcz_map_engine - INFO - Streaming COG and clipping to geometry...


13:34:37 - rasterio._env - WARNING - CPLE_IllegalArg in tmps9abxca2.tif: BLOCKXSIZE can only be used with TILED=YES


13:34:37 - LCZ4py._internal._lcz_map_engine - INFO - Saved to clipped cache.


/Users/co2map/.lcz4r_cache/clipped_c7031cb44f08aaf6.tif


## 2. `lcz_get_ucp`: assinatura e o que você recebe de volta

```python
def lcz_get_ucp(
    lcz_map,                      # caminho/DataArray que define a área de estudo
    stations=None,                # None -> pilha raster; GeoDataFrame/DataFrame/caminho -> extração pontual
    cache_dir="lcz4r_cache",      # cache em disco para rasters baixados/processados
    n_workers=None,                # workers paralelos de download (padrão: nº de CPUs - 1)
    verbose=True,
    variables=None,                # subconjunto de variáveis, ex.: ["elevation", "tree", "lf_0"]
    ghsl_tiles=None,                # IDs específicos de tiles GHSL (None = detecção automática pela extensão de lcz_map)
    process_ghsl=True,             # superfície/altura/volume construído + população (GHSL)
    process_wumpod=True,           # morfologia de edificações + cobertura do solo (WUDAPT/WUMPOD)
    process_vegetation=True,       # cobertura arbórea / superfície impermeável (GLC_FCS30D)
    process_directional=True,      # rugosidade de cânion urbano em 0/45/90/135 graus
    use_threads=True,              # deve permanecer True (ver abaixo)
    fail_fast=False,                # se True, lança erro na primeira variável que falhar em vez de pular
) -> dict
```

Internamente, isso encapsula um `UrbanParameterProcessor` que baixa cada variável solicitada **em
paralelo** via streaming `/vsicurl/` do GDAL (então, como as demais funções `lcz_get_*` deste pacote,
apenas os pixels que intersectam a extensão do seu `lcz_map` são transferidos, nunca rasters globais
completos), reamostra tudo para uma grade comum e, em seguida, (a) extrai valores nas coordenadas das
estações ou (b) monta o resultado como uma pilha raster. `use_threads=False` propositalmente lança um
erro — o processador mantém uma `requests.Session` e um `diskcache.FanoutCache` em `self`, nenhum dos
quais sobrevive a ser serializado (pickled) para um processo separado, e como esses downloads são
limitados por I/O, threads já os paralelizam com eficiência.

**Valor de retorno** — um `dict` simples (não um dataclass, diferente da maioria das outras funções
`lcz_get_*`), pois seu formato depende de `stations` ter sido informado ou não:

- `'df_vars'` — um `pandas.DataFrame` com uma linha por estação e uma coluna por variável processada
  com sucesso, mais a coluna identificadora (`None` se `stations=None`).
- `'combined_rasters'` — um `xarray.Dataset` contendo cada variável processada como seu próprio
  `DataArray` 2D, já reamostrado/alinhado à grade de `lcz_map`.
- `'stack_path'` — caminho para a pilha GeoTIFF salva `LCZ4r_output/lcz_ucp_stack.tif` (apenas quando
  `stations=None`; caso contrário `None`, pois nesse caso a tabela por estação é o produto final).
- `'variable_list'` — lista de nomes de variáveis baixadas e processadas com sucesso.
- `'failed_variables'` — lista de tuplas `(nome_da_variavel, mensagem_de_erro)` para qualquer variável
  que falhou (instabilidades de rede, tiles ausentes, etc. — o processador é tolerante a falhas por
  padrão: `fail_fast=False` significa que uma variável falha não interrompe toda a execução).
- `'summary'` — um dataclass `ProcessingSummary`: `total_variables`, `successful`, `failed`,
  `total_time` (segundos), `failed_variables`.

Vamos exercitar os dois ramos abaixo: primeiro `stations=None` (modo pilha raster), depois
`stations=<alguns pontos>` (modo de extração por estação, o que importa para o fluxo de ML de
interpolação em local/05).

### 2a. Modo pilha raster (`stations=None`)

Em uma primeira passagem, restringimos `variables` a um punhado de parâmetros baratos e ilustrativos
de cada família não-GHSL — `elevation`, `tree` (vegetação), `urban` (superfície impermeável) e três
métricas de rugosidade direcional (`lf_0`, `hi_45`, `zdm_90`) — e desativamos `process_ghsl` e a
varredura completa de `process_directional`. Isso mantém o download minúsculo (um punhado de pequenos
rasters recortados em vez de dezenas) enquanto ainda mostra todas as famílias de UCP exceto GHSL em
ação. O GHSL é tratado separadamente na seção 4 abaixo porque — diferente das fontes WUDAPT/WUMPOD e
de vegetação, que são servidas como GeoTIFFs Cloud-Optimized simples — os tiles do GHSL são
distribuídos como arquivos zipados, então `/vsicurl/` precisa transitar também por `/vsizip/`, o que é
consideravelmente mais lento mesmo para um único tile pequeno (cada variável do GHSL levou de 30 a 125
segundos para o tile único de Juiz de Fora nos testes, contra poucos segundos para as variáveis WUMPOD/
vegetação/direcionais abaixo).

In [3]:
from LCZ4py.general.lcz_get_ucp import lcz_get_ucp

result_stack = lcz_get_ucp(
    lcz_map=lcz_map_path,
    stations=None,
    n_workers=4,
    variables=["elevation", "tree", "urban", "lf_0", "hi_45", "zdm_90"],
    process_ghsl=False,
    process_wumpod=True,
    process_vegetation=True,
    process_directional=True,
)

print(result_stack["variable_list"])
print(result_stack["failed_variables"])
print(result_stack["summary"])
print(result_stack["stack_path"])
result_stack["combined_rasters"]

13:34:37 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


13:34:37 - LCZ4py.general.lcz_get_ucp - INFO - Urban Parameter Processor Initialized


13:34:37 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


13:34:37 - LCZ4py.general.lcz_get_ucp - INFO - Study Area ID: -43.686_-22.0_-43.146_-21.52


13:34:37 - LCZ4py.general.lcz_get_ucp - INFO - Target CRS: EPSG:4326


...

13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - Raster layers: 6


13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - Stack saved: LCZ4r_output/lcz_ucp_stack.tif


13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


['lf_0', 'elevation', 'urban', 'hi_45', 'tree', 'zdm_90']
[]
ProcessingSummary(total_variables=6, successful=6, failed=0, total_time=22.64597773551941, failed_variables=[])
LCZ4r_output/lcz_ucp_stack.tif


<xarray.Dataset> Size: 15MB
Dimensions:      (x: 601, y: 534)
Coordinates:
  * x            (x) float64 5kB -43.69 -43.68 -43.68 ... -43.15 -43.15 -43.15
  * y            (y) float64 4kB -21.52 -21.52 -21.52 ... -22.0 -22.0 -22.0
    spatial_ref  int64 8B 0
Data variables:
    lf_0         (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan
    elevation    (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan
    urban        (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan
    hi_45        (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan
    tree         (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan
    zdm_90       (y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan nan

`result_stack['summary']` informa quantas das variáveis solicitadas tiveram sucesso/falharam e o
tempo total decorrido; com o cache de rede já aquecido (como estará após a primeira execução deste
notebook), isso termina em poucos segundos. `result_stack['combined_rasters']` é um `xarray.Dataset` —
um `DataArray` por variável, já recortado e reamostrado para a mesma grade de `lcz_map_path` — e
`stack_path` é onde esses mesmos dados foram gravados em disco como um GeoTIFF multibanda
(`lcz_ucp_stack.tif`, uma banda por variável, com as descrições de banda definidas como o nome da
variável), já que nenhuma estação foi fornecida. Essa pilha raster é exatamente o formato que
`lcz_plot_parameters` e `lcz_interp_map_plus`/`lcz_interp_eval_plus` (notebook
`local/05_ml_interpolation_ucp`) esperam como fonte de variáveis preditoras em grade.

### 3. Visualizando a pilha raster de UCPs

`lcz_plot_parameters` aceita diretamente o `dict` retornado por `lcz_get_ucp(..., stations=None)` —
sem necessidade de recarregá-lo do disco. Passar `all_params=True` plota cada variável de
`combined_rasters` como seu próprio mapa interativo.

In [4]:
from LCZ4py.general.lcz_plot_parameters import lcz_plot_parameters

figs = lcz_plot_parameters(result_stack, all_params=True)
print(type(figs), len(figs))
figs[1]

<class 'list'> 6


Cada painel é um raster contínuo sobre a área de Juiz de Fora — por exemplo, `tree` (% de cobertura
arbórea) deve ser visivelmente maior sobre as encostas florestadas e menor no fundo de vale construído,
enquanto `urban` (% de superfície impermeável) mostra o padrão oposto. Como esses são valores diretamente
observados, pixel a pixel (não valores inferidos de uma tabela de referência de classe LCZ), você verá
aqui heterogeneidade real dentro da mesma classe, que os parâmetros analíticos de `lcz_get_parameters`
não conseguem mostrar — dois pixels na mesma classe LCZ podem, e frequentemente têm, cobertura arbórea
diferente.

### 4. Modo de extração por estação (`stations=...`)

Agora o modo mais importante para conectar UCPs à análise climática baseada em estações: passe um
`GeoDataFrame` de locais pontuais e `lcz_get_ucp` extrai o valor de cada variável processada em cada
ponto, em vez de retornar um raster. Em um fluxo real, esses seriam as coordenadas das suas estações
meteorológicas (por exemplo, as colunas `Latitude`/`Longitude` de `lcz_data_berlin.csv`, usadas ao
longo de toda a série de notebooks `local/`); aqui usamos três pontos ilustrativos espalhados pela
pequena área de Juiz de Fora para manter o exemplo rápido e autocontido.

In [5]:
import geopandas as gpd
from shapely.geometry import Point

stations = gpd.GeoDataFrame(
    {"station": ["S1", "S2", "S3"]},
    geometry=[Point(9.52, 47.13), Point(9.55, 47.14), Point(9.58, 47.15)],
    crs="EPSG:4326",
)

result_stations = lcz_get_ucp(
    lcz_map=lcz_map_path,
    stations=stations,
    n_workers=4,
    variables=["elevation", "tree", "urban"],
    process_ghsl=False,
    process_wumpod=True,
    process_vegetation=True,
    process_directional=False,
)

result_stations["df_vars"]

13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - Urban Parameter Processor Initialized


13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - Study Area ID: -43.686_-22.0_-43.146_-21.52


13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - Target CRS: EPSG:4326


...

13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - ⏱️  Total Time: 0.2s


13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - Raster layers: 3


13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - DataFrame: 3 rows × 4 cols


13:35:00 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


,station,elevation,tree,urban
0,S1,NaN,NaN,NaN
1,S2,NaN,NaN,NaN
2,S3,NaN,NaN,NaN


`df_vars` agora é uma tabela por estação: uma linha por estação, uma coluna por variável processada
com sucesso, unida pela coluna identificadora da estação. Essa é a tabela que você alimentaria
diretamente em `lcz_interp_map_plus`/`lcz_interp_eval_plus` (`ucp_variables=...`) como matriz de
atributos de um modelo Random Forest de interpolação de temperatura, ou que você juntaria às suas
observações de temperatura do ar para regredir "quanto da diferença de temperatura entre a estação A e
a estação B é explicado pela cobertura arbórea / fração impermeável / altura de edificação" —
transformando uma explicação puramente categórica baseada em LCZ em uma explicação contínua e
quantitativa. `stack_path` é `None` aqui (como documentado acima), já que a pilha raster não é o
produto final quando estações são fornecidas.

## 5. Densidade/altura/volume/população construídos GHSL (referência — não executado aqui)

`process_ghsl=True` baixa e monta adicionalmente quatro camadas GHSL — `built_surface`,
`built_height`, `built_volume`, `pop` — detectando automaticamente quais células da grade global de
tiles do GHSL intersectam a extensão do seu `lcz_map` (ou aceitando uma lista explícita
`ghsl_tiles=[...]`). O código abaixo é exatamente o que você executaria:

```python
result_ghsl = lcz_get_ucp(
    lcz_map=lcz_map_path,
    stations=None,
    n_workers=4,
    process_ghsl=True,
    process_wumpod=False,
    process_vegetation=False,
    process_directional=False,
)
print(result_ghsl["variable_list"], result_ghsl["failed_variables"])
```

**Esta célula foi deliberadamente deixada sem execução.** Nos testes, mesmo para o único tile GHSL
minúsculo de Juiz de Fora, cada uma das quatro variáveis levou de 30 segundos a mais de 2 minutos para
transmitir (os tiles do GHSL são servidos como arquivos zipados, então o GDAL precisa abri-los via uma
cadeia `/vsizip//vsicurl/` em vez do simples `/vsicurl/`, o que é marcadamente mais lento que as fontes
WUDAPT/WUMPOD/vegetação/direcionais acima) — cerca de 4-5 minutos no total para quatro variáveis e um
tile. Para uma cidade maior, abrangendo múltiplos tiles do GHSL, ou quando vários notebooks/agentes
compartilham o mesmo caminho de rede, isso facilmente ultrapassa o que é razoável embutir na execução
automatizada de um tutorial. Se você executar isso por conta própria, espere que leve vários minutos
mesmo para uma cidade pequena, e reserve tempo proporcionalmente maior para uma em escala metropolitana;
o ganho é o mesmo conjunto de camadas de ambiente construído — de zero a densidade populacional — pelo
qual o GHSL é amplamente usado em estudos de clima urbano e exposição.

## 6. Rugosidade direcional de cânion urbano, varredura completa (referência)

A seção 2a já exercitou três variáveis direcionais (`lf_0`, `hi_45`, `zdm_90`) para provar que o
mecanismo funciona e é rápido. A varredura *completa* `process_directional=True` (padrão do pacote)
calcula fração de terreno e índice de altura em cada uma de quatro direções, além de quatro parâmetros
de rugosidade aerodinâmica (`zdm`, `zdr`, `zom`, `zor`) em cada direção — cerca de 30 variáveis no
total:

```python
result_directional = lcz_get_ucp(
    lcz_map=lcz_map_path,
    stations=None,
    n_workers=8,
    process_ghsl=False,
    process_wumpod=False,
    process_vegetation=False,
    process_directional=True,   # ~30 variáveis: lf_*, hi_*, zdm_*, zdr_*, zom_*, zor_* em 0/45/90/135°
)
```

Cada uma dessas variáveis baixa quase tão rápido quanto as três que já executamos (poucos segundos, já
que vêm da mesma fonte Zenodo de GeoTIFF simples que o WUMPOD, não dos arquivos GHSL zipados mais
lentos), então a varredura completa custa principalmente mais *variáveis*, não mais *tempo por
variável* — com `n_workers` aumentado proporcionalmente, é razoável executá-la você mesmo para uma
caracterização completa da morfologia urbana, mas mantemos a demonstração executada acima restrita ao
subconjunto de três variáveis para manter o tempo total de execução deste notebook curto.

## Conclusão e próximos passos

`lcz_get_ucp` conecta a forma urbana derivada de sensoriamento remoto (densidade/altura/volume
construído/população do GHSL, morfologia de edificações WUDAPT/WUMPOD, cobertura vegetal/impermeável
do GLC_FCS30D e rugosidade direcional de cânion urbano) com dois modos de consumo: uma pilha raster em
grade para mapeamento e visualização de toda a cidade, ou extração pontual em coordenadas específicas
— o mais importante, locais de estações meteorológicas — para atribuir diferenças de temperatura em
nível de estação à forma urbana física que de fato envolve cada sensor. Essa tabela de UCPs por estação
é precisamente o conjunto de atributos usado para treinar o notebook de interpolação espacial por
machine learning.

**Anterior:** [`05_spectral_indices`](./05_spectral_indices.pt.ipynb) — NDVI/NDBI/MNDWI/EVI por classe
LCZ a partir do Sentinel-2.

**Próximo:** [`07_gridded_climate_environment`](./07_gridded_climate_environment.pt.ipynb) — download
de variáveis climáticas em grade (ERA5, CHIRPS) e ambientais (poluição, seca) recortadas para um mapa
LCZ, o último notebook da série `general/` antes da série baseada em estações `local/`
([`local/01_lcz_time_series`](../local/01_lcz_time_series.pt.ipynb)), onde a tabela de UCPs por estação
produzida aqui alimenta diretamente
[`local/05_ml_interpolation_ucp`](../local/05_ml_interpolation_ucp.pt.ipynb).